# TechCorp — Fine-tuning médical LoRA (QLoRA)

**Modèle expérimental** — ne pas déployer en production.

Dataset : [ruslanmv/ai-medical-chatbot](https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot)

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets trl sentencepiece

In [ ]:
import json, re, torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"
MAX_SAMPLES = 2000

def format_phi3(instr, resp):
    return f"<|user|>\n{instr.strip()}<|end|>\n<|assistant|>\n{resp.strip()}<|end|>"

ds = load_dataset("ruslanmv/ai-medical-chatbot", split="train")
records = []
for row in ds:
    desc = str(row.get("Description") or "").strip()
    patient = str(row.get("Patient") or "").strip()
    doctor = str(row.get("Doctor") or "").strip()
    instr = f"{desc}\n{patient}".strip() if patient else desc
    if len(instr) < 15 or len(doctor) < 15:
        continue
    records.append({"text": format_phi3(instr, doctor)})
    if len(records) >= MAX_SAMPLES:
        break

split = int(len(records) * 0.9)
train_records, val_records = records[:split], records[split:]
print(f"Train: {len(train_records)}, Val: {len(val_records)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map="auto", trust_remote_code=True)
model = prepare_model_for_kbit_training(model)

lora = LoraConfig(task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512, padding="max_length")

train_ds = Dataset.from_list(train_records).map(tok, batched=True, remove_columns=["text"])
val_ds = Dataset.from_list(val_records).map(tok, batched=True, remove_columns=["text"])

args = TrainingArguments(
    output_dir="./medical_lora",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

history = trainer.train()
metrics = trainer.evaluate()
print({"train_loss": history.training_loss, "eval_loss": metrics.get("eval_loss")})

In [ ]:
model.save_pretrained("./medical_lora")
tokenizer.save_pretrained("./medical_lora")

# Test rapide
q = "Quels sont les symptômes de l'hypertension ?"
prompt = f"<|user|>\n{q}<|end|>\n<|assistant|>\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0], skip_special_tokens=True))